# 🦷 Dental Index Prediction — Training Notebook

**Multi-view EfficientNet (default B3) with CORAL ordinal regression** — prefers GPU, auto-falls back to CPU, uses mixed precision on CUDA.

Outputs 3 indices from 3 views:
- **MGI** 0–4
- **OHI** 0–3
- **GEI** 0–2

Backbone is configurable via `CONFIG['backbone_name']` (default `efficientnet_b3`).

## 1. Setup & Configuration

In [ ]:
import os
import sys
import json
import time
import random
import numpy as np
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

import matplotlib.pyplot as plt
import matplotlib
from tqdm.notebook import tqdm

PROJECT_ROOT = Path(r'E:\Dental_Project')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.dataset import DentalDataset, get_class_weights
from ml.model import MultiViewDentalModel

matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['figure.dpi'] = 110

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# ==================== CONFIGURATION ====================
CONFIG = {
    'data_dir': str(PROJECT_ROOT / 'Thesis_Data'),
    'output_dir': str(PROJECT_ROOT / 'ml' / 'checkpoints'),
    'epochs': 60,
    'batch_size': 4,
    'lr': 1e-4,
    'weight_decay': 1e-4,
    'patience': 12,
    'seed': 42,
    'val_ratio': 0.2,
    'unfreeze_epoch': 10,
    'backbone_lr_mult': 0.1,
    'backbone_name': 'efficientnet_b3',  # robust default; change to b4/b5 if GPU is strong
    'image_size': 300,                    # match backbone input
    
    'resume_from': None,
    

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
scaler = GradScaler(enabled=(device.type == 'cuda'))
os.makedirs(CONFIG['output_dir'], exist_ok=True)
print(f"Using device: {device}")

## 2. Transforms (image size driven by config)

In [ ]:
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def get_train_transforms(img_size):
    return transforms.Compose([
        transforms.Resize((img_size + 24, img_size + 24)),
    
        transforms.RandomHorizontalFlip(p=0.5),
    
        transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.2, hue=0.05),
    
0.05
1.06
        transforms.ToTensor(),
    
        transforms.RandomErasing(p=0.1, scale=(0.02, 0.12)),
    

## 3. Dataset Exploration

In [ ]:
full_dataset = DentalDataset(CONFIG['data_dir'])
print(f"Total complete samples: {len(full_dataset)}")

mgi_dist = np.bincount([s['mgi'] for s in full_dataset.samples], minlength=5)
ohi_dist = np.bincount([s['ohi'] for s in full_dataset.samples], minlength=4)
gei_dist = np.bincount([s['gei'] for s in full_dataset.samples], minlength=3)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors_mgi = ['#22c55e', '#84cc16', '#eab308', '#f97316', '#ef4444']
colors_ohi = ['#22c55e', '#84cc16', '#eab308', '#ef4444']
colors_gei = ['#22c55e', '#eab308', '#ef4444']
axes[0].bar(range(5), mgi_dist, color=colors_mgi, edgecolor='white', linewidth=1.4)
axes[1].bar(range(4), ohi_dist, color=colors_ohi, edgecolor='white', linewidth=1.4)
axes[2].bar(range(3), gei_dist, color=colors_gei, edgecolor='white', linewidth=1.4)
axes[0].set_title('MGI Distribution'); axes[1].set_title('OHI Distribution'); axes[2].set_title('GEI Distribution')
for ax, dist in zip(axes, [mgi_dist, ohi_dist, gei_dist]):
    for i, v in enumerate(dist):
        ax.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Data Loading & Splitting

In [ ]:
all_ids = [s['sl_no'] for s in full_dataset.samples]
mgi_labels = {s['sl_no']: s['mgi'] for s in full_dataset.samples}

class_groups = defaultdict(list)
for pid in all_ids:
    class_groups[mgi_labels[pid]].append(pid)

rng = np.random.RandomState(CONFIG['seed'])
train_ids, val_ids = [], []
for cls, pids in class_groups.items():
    rng.shuffle(pids)
    n_val = max(1, int(len(pids) * CONFIG['val_ratio']))
    val_ids.extend(pids[:n_val])
    train_ids.extend(pids[n_val:])

print(f"Train: {len(train_ids)}")
print(f"Val:   {len(val_ids)}")

train_dataset = DentalDataset(CONFIG['data_dir'], transform=get_train_transforms(CONFIG['image_size']), patient_ids=train_ids)
val_dataset = DentalDataset(CONFIG['data_dir'], transform=get_val_transforms(CONFIG['image_size']), patient_ids=val_ids)

mgi_labels_train = [s['mgi'] for s in train_dataset.samples]
class_counts = np.bincount(mgi_labels_train, minlength=5)
class_weights_sampler = 1.0 / (class_counts + 1e-6)
sample_weights = [class_weights_sampler[label] for label in mgi_labels_train]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], sampler=sampler, num_workers=0, pin_memory=(device.type=='cuda'), drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0, pin_memory=(device.type=='cuda'))

print(f"Train batches/epoch: {len(train_loader)}")
print(f"Val batches/epoch:   {len(val_loader)}")

## 5. Model & Loss (GPU-first, CPU fallback)

In [ ]:
class_weights = get_class_weights(CONFIG['data_dir'])
print('Class weights:')
for k,v in class_weights.items():
    print(f"  {k}: {v}")

model = MultiViewDentalModel(
    backbone_name=CONFIG['backbone_name'],
    pretrained=True,
    dropout=CONFIG['dropout'],
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}")
print(f"Trainable params (before freeze): {trainable_params:,}")

for p in model.backbone.parameters():
    p.requires_grad = False
print(f"Trainable after freezing backbone: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

from ml.losses import MultiTaskOrdinalLoss
criterion = MultiTaskOrdinalLoss(
    mgi_weights=class_weights['mgi'],
    ohi_weights=class_weights['ohi'],
    gei_weights=class_weights['gei'],
).to(device)

head_params = (
    list(model.fusion_layers.parameters()) +
    list(model.mgi_head.parameters()) +
    list(model.ohi_head.parameters()) +
    list(model.gei_head.parameters())
)
optimizer = torch.optim.AdamW(head_params, lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

print('Optimizer: AdamW with cosine restarts')

## 6. (Optional) Resume from Checkpoint

In [ ]:
start_epoch = 1
best_val_loss = float('inf')
history = []

if CONFIG['resume_from'] and os.path.exists(CONFIG['resume_from']):
    print(f"Resuming from {CONFIG['resume_from']}")
    checkpoint = torch.load(CONFIG['resume_from'], map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    if 'optimizer_state_dict' in checkpoint:
        # Prepare correct param groups depending on unfreeze state
        if checkpoint['epoch'] >= CONFIG['unfreeze_epoch']:
            for p in model.backbone.parameters():
                p.requires_grad = True
            optimizer = torch.optim.AdamW([
                {'params': model.backbone.parameters(), 'lr': CONFIG['lr'] * CONFIG['backbone_lr_mult']},
                {'params': head_params, 'lr': CONFIG['lr']},
            ], weight_decay=CONFIG['weight_decay'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_loss = checkpoint.get('val_loss', float('inf'))
    history = checkpoint.get('history', [])
    print(f"Resumed at epoch {start_epoch}, best val_loss {best_val_loss:.4f}")
else:
    print('Starting fresh training')

## 7. Training Helpers (with AMP on GPU)

In [ ]:
def compute_accuracy(outputs, batch):
    accs = {}
    for key in ['mgi', 'ohi', 'gei']:
        logits = outputs[key]
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).sum(dim=1).int()
        labels = batch[f'{key}_label']
        accs[key] = (preds == labels).float().mean().item()
    return accs

def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch, pbar=None):
    model.train()
    total_loss = 0.0
    acc_sum = {'mgi':0, 'ohi':0, 'gei':0}
    steps = 0

    for batch in loader:
        frontal = batch['frontal'].to(device)
        left = batch['left_lateral'].to(device)
        right = batch['right_lateral'].to(device)
        for k in ['mgi_target','ohi_target','gei_target','mgi_label','ohi_label','gei_label']:
            batch[k] = batch[k].to(device)

        with autocast(enabled=(device.type=='cuda')):
            outputs = model(frontal, left, right)
            loss, losses = criterion(outputs, batch)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += losses['total']
        acc = compute_accuracy(outputs, batch)
        for k in acc_sum: acc_sum[k] += acc[k]
        steps += 1

        if pbar: pbar.update(1); pbar.set_postfix({'loss': f
})

    return total_loss/steps, {k: v/steps for k,v in acc_sum.items()}

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    acc_sum = {'mgi':0,'ohi':0,'gei':0}
    steps = 0
    for batch in loader:
        frontal = batch['frontal'].to(device)
        left = batch['left_lateral'].to(device)
        right = batch['right_lateral'].to(device)
        for k in ['mgi_target','ohi_target','gei_target','mgi_label','ohi_label','gei_label']:
            batch[k] = batch[k].to(device)
        with autocast(enabled=(device.type=='cuda')):
            outputs = model(frontal, left, right)
            loss, losses = criterion(outputs, batch)
        total_loss += losses['total']
        acc = compute_accuracy(outputs, batch)
        for k in acc_sum: acc_sum[k] += acc[k]
        steps += 1
    if steps == 0: return 0, {k:0 for k in acc_sum}
    return total_loss/steps, {k: v/steps for k,v in acc_sum.items()}

print('Helpers ready (AMP enabled on CUDA)')

## 8. Training Loop (with unfreeze + early stopping)

In [ ]:
patience_counter = 0
epochs = CONFIG['epochs']
unfreeze_epoch = CONFIG['unfreeze_epoch']

print(f"{'='*70}")
print(f"Training for {epochs} epochs; backbone unfreezes after {unfreeze_epoch}")
print(f"Patience: {CONFIG['patience']}")
print(f"{'='*70}")

epoch_pbar = tqdm(range(start_epoch, epochs + 1), desc='Epochs', unit='epoch')

for epoch in epoch_pbar:
    if epoch == unfreeze_epoch + 1:
        print(f"\n>>> Unfreezing backbone at epoch {epoch}")
        for p in model.backbone.parameters():
            p.requires_grad = True
        optimizer = torch.optim.AdamW([
            {'params': model.backbone.parameters(), 'lr': CONFIG['lr'] * CONFIG['backbone_lr_mult']},
            {'params': head_params, 'lr': CONFIG['lr']},
        ], weight_decay=CONFIG['weight_decay'])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
        print(f"Trainable params now: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    start_t = time.time()
    batch_pbar = tqdm(total=len(train_loader), desc=f'Epoch {epoch}/{epochs}', leave=False, unit='batch')
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, epoch, pbar=batch_pbar)
    batch_pbar.close()

    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()

    elapsed = time.time() - start_t
    history.append({
        'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
        'train_acc': train_acc, 'val_acc': val_acc,
    